# Customizing the Eval Schema

In `01_example_simple`, the default eval schema scored everything with `exact` (strings) and `numeric` (numbers). That caused:
- "CVD" vs "Chemical Vapor Deposition" → mismatch (synonyms not recognized)
- 302 vs 300 → mismatch (no tolerance)

This notebook shows how to fix these by editing the eval schema:
1. **oneof** comparator for known synonyms
2. **numeric tolerance** for approximate number matching
3. **transforms** for normalization before comparison
4. **x-eval-skip** for fields you don't want to score
5. **custom comparator** for comparison logic the package doesn't provide

For global/process-wide settings (like changing the default comparator for an entire type via `set_type_default()`), see the README.

In [1]:
GOLD = [
    {
        "material": "silicon",
        "temperature": 300.0,
        "method": "Chemical Vapor Deposition",
        "date_created": "2024-01-15",
        "note": "CVD on silicon",
    },
    {
        "material": "Gold",
        "temperature": 200.0,
        "method": "Evaporation",
        "date_created": "2024-06-01",
        "note":"",
    },
]

EXTRACTED = [
    {
        "material": "Silicon  ", # capital, space
        "temperature": 302.0,
        "method": "CVD",
        "date_created": "Jan 15, 2024",     # different format
        "note":"",
    },
    {
        "material": "Gold",
        "temperature": 200.0,
        "method": "Evaporation",
        "date_created": "Jun 01, 2024",      # different format
        "note":"",
    },
]

## Start with the Default Schema

In this notebook we edit the schema in Python for demonstration. In practice,
you'd **save the schema to a JSON file**, edit it in your editor, then load it back:

```python
import json
eval_schema = infer_schema(GOLD)
annotate_xeval(eval_schema)
with open("eval_schema.json", "w") as f:
    json.dump(eval_schema, f, indent=2)

# --- open eval_schema.json in your editor, customize x-eval-* keys ---

# Load back
with open("eval_schema.json") as f:
    eval_schema = json.load(f)
result = evaluate(GOLD, EXTRACTED, schema=eval_schema)
```

The schema file is the single source of truth for evaluation config. Version-control it alongside your gold data.

In [2]:
import json
from copy import deepcopy
from struct_extract_eval import infer_schema, annotate_xeval, evaluate

eval_schema = infer_schema(GOLD)
annotate_xeval(eval_schema)
print(json.dumps(eval_schema, indent=2))

{
  "type": "object",
  "properties": {
    "date_created": {
      "type": "string",
      "x-eval-compare": "exact"
    },
    "material": {
      "type": "string",
      "x-eval-compare": "exact"
    },
    "method": {
      "type": "string",
      "x-eval-compare": "exact"
    },
    "note": {
      "type": "string",
      "x-eval-compare": "exact"
    },
    "temperature": {
      "type": "number",
      "x-eval-compare": "numeric"
    }
  }
}


## Fix 1: Synonyms with `oneof`

"CVD" and "Chemical Vapor Deposition" are the same method. The `oneof` comparator
matches if the extracted value is in a list of accepted values.

In [3]:
schema = deepcopy(eval_schema)

# Replace exact with oneof for the method field
schema["properties"]["method"]["x-eval-compare"] = {
    "oneof": {
        "values": [
            "Chemical Vapor Deposition", "CVD",
            "Sputtering", "PVD", "Physical Vapor Deposition",
            "Evaporation", "Thermal Evaporation",
        ]
    }
}

run = evaluate(GOLD, EXTRACTED, schema=schema)
method = run.per_field["method"]
print(f"method: score={method.mean_score:.2f}")

method: score=1.00


## Fix 2: Numeric Tolerance

Temperature 302 vs 300 is within 1% relative tolerance. Add `rel` tolerance to the
`numeric` comparator.

In [4]:
schema = deepcopy(eval_schema)

# Add 1% relative tolerance to temperature
schema["properties"]["temperature"]["x-eval-compare"] = {
    "numeric": {"tolerance": {"rel": 0.01}}
}

run = evaluate(GOLD, EXTRACTED, schema=schema)
temp = run.per_field["temperature"]
print(f"temperature: score={temp.mean_score:.2f}") # temperature 302 degree is within the 1% tolerance error of 300 degree

temperature: score=1.00


## Fix 3: Transforms

Transforms preprocess BOTH gold and extracted values before comparison.
Common use: `lowercase` + `strip` for case-insensitive matching.

In [5]:
schema = deepcopy(eval_schema)

# Add lowercase + strip transforms to material
schema["properties"]["material"]["x-eval-transform"] = ["lowercase", "strip"]


run = evaluate(GOLD, EXTRACTED, schema=schema)
mat = run.per_field["material"]
print(f"material: score={mat.mean_score:.2f}")

material: score=1.00


## Fix 4: Skip Fields

Some fields have no correct answer (e.g., free-text descriptions).
Mark them `x-eval-skip: true` to exclude from scoring.

In [6]:
# Mark the "note" field as skip -- excluded from scoring
schema = deepcopy(eval_schema)
schema["properties"]["note"] = {
    "type": "string",
    "x-eval-skip": True,
}

run = evaluate(GOLD, EXTRACTED, schema=schema)
print(f"Total fields scored: {run.total_fields}") # note field is excluded from scoring

Total fields scored: 8


## Fix 5: Custom Comparator

When the built-in comparators don't cover your
use case, write your own and register it. A comparator is a function that takes
`(gold, extracted, params)` and returns a `ComparatorResult`.

Register it by name, then reference that name in the schema with `x-eval-compare`.

**Example:** a date comparator that handles multiple date formats.

In [7]:
from datetime import datetime
from struct_extract_eval.core.comparators.comparator import ComparatorResult
from struct_extract_eval.core.comparators.registry import register


def compare_date(gold, extracted, params):
    """Compare dates regardless of format."""
    formats = params.get("formats", ["%Y-%m-%d", "%b %d, %Y", "%d/%m/%Y"])

    def parse(val):
        for fmt in formats:
            try:
                return datetime.strptime(str(val), fmt)
            except ValueError:
                continue
        return None

    g, e = parse(gold), parse(extracted)
    score = 1.0 if (g and e and g == e) else 0.0
    return ComparatorResult(score=score, comparator="date_comp")


# Register it -- overwrite=True makes this safe for notebook re-runs
register("date_comp", compare_date, overwrite=True)

# Test it
schema = deepcopy(eval_schema)
schema["properties"]["date_created"] = {
    "type": "string",
    "x-eval-compare": {"date_comp": {"formats": ["%Y-%m-%d", "%b %d, %Y"]}},
}


run = evaluate(GOLD, EXTRACTED, schema=schema)
date_field = run.per_field["date_created"]
print(f"date_created: score={date_field.mean_score:.2f}") # '2024-01-15' matches 'Jan 15, 2024'
print("  '2024-01-15' vs 'Jan 15, 2024' -> same date -> match!")

date_created: score=1.00
  '2024-01-15' vs 'Jan 15, 2024' -> same date -> match!


## Fix 6: A Comparator on a Container Node

So far every `x-eval-compare` sat on a leaf. It can also sit on an **object or array node**.
That node is then scored as one unit: the comparator receives the whole value on each side
(the full dict or list), decides however it likes, and returns **one** result for the node.
Its children are never scored individually, and their `x-eval-*` keys are ignored.

Use it when the unit of correctness is the whole object, or when the parts you want to compare
together live at different depths.


In [8]:
from example_utils import show_run

# A nested record: the person is one unit, even though its parts are at different depths.
GOLD_P = [{"person": {"full_name": {"surname": "Smith", "name": "John"}, "nickname": "Johnny"}}]
EXTRACTED_P = [{"person": {"full_name": {"surname": "Smith", "name": "John"}, "nickname": "JOHNNY"}}]  # only the case differs


def compare_person(gold, extracted, params):
    # gold / extracted are the complete dicts under "person"
    if gold["full_name"] != extracted["full_name"]:
        return ComparatorResult(score=0.0, comparator="person", reason="name differs")
    g_nick = (gold.get("nickname") or "").lower()
    e_nick = (extracted.get("nickname") or "").lower()
    if g_nick != e_nick:
        return ComparatorResult(score=0.0, comparator="person", reason="nickname differs")
    return ComparatorResult(score=1.0, comparator="person")


register("person", compare_person, overwrite=True)

schema_p = infer_schema(GOLD_P)
schema_p["properties"]["person"]["x-eval-compare"] = "person"   # on the object node, not a leaf
annotate_xeval(schema_p)

show_run(evaluate(GOLD_P, EXTRACTED_P, schema=schema_p), "Comparator on the 'person' object", values=False)


Comparator on the 'person' object
  mean P=1.00  R=1.00  F1=1.00   (1 record(s))
  record  path    score  status  reason
  0       person  1.0    match


One row per record for `person`, whatever happens inside it. Three consequences:

- `person` counts as **one field** in precision, recall and F1: a wrong object is one mismatch,
  a missing object is one omission.
- The comparator gets **raw values**; normalisation (like the `.lower()` above) belongs inside it.
- If you only want to couple *some* siblings and keep the rest individually scored, use a
  compound comparator instead -- see the end of `05_example_compound`.


## Putting It All Together

Apply all fixes to the original data. Before evaluating:
1. **`parse_eval_schema()`** — validate the eval schema itself (catches typos in comparator names, malformed `x-eval-*` config)
2. **`validate_gold()`** — validate gold data against the schema (catches wrong types)

In [9]:
from struct_extract_eval import parse_eval_schema, validate_gold

# Step 1: Build the fully customized schema
schema = infer_schema(GOLD)
# step 2: add x-eval-* keys to customize comparison logic
annotate_xeval(schema)

# step 3: Customize eval schema
schema["properties"]["method"]["x-eval-compare"] = {
    "oneof": {"values": [
        "Chemical Vapor Deposition", "CVD",
        "Sputtering", "PVD",
        "Evaporation", "Thermal Evaporation",
    ]}
}
schema["properties"]["temperature"]["x-eval-compare"] = {
    "numeric": {"tolerance": {"rel": 0.01}}
}
schema["properties"]["material"]["x-eval-transform"] = ["lowercase", "strip"]

schema["properties"]["note"] = {
    "type": "string",
    "x-eval-skip": True,
}

schema["properties"]["date_created"] = {
    "type": "string",
    "x-eval-compare": {"date_comp": {"formats": ["%Y-%m-%d", "%b %d, %Y"]}},
}

# step 4: Validate the schema -- catches x-eval-* config errors
parse_eval_schema(schema)
print("Schema validation passed.")

# Step 5: Validate gold -- warn the gold field is not in schema, or the gold value is not the type declared in the schema
validate_gold(GOLD, schema)
print("Gold validation passed.")

# step 6: Evaluate
run_before = evaluate(GOLD, EXTRACTED, schema=eval_schema)
run_after = evaluate(GOLD, EXTRACTED, schema=schema)

print(f"\n{'Metric':<20} {'Before':>8} {'After':>8}")
print("-" * 36)
print(f"{'Precision':<20} {run_before.mean_precision:>8.3f} {run_after.mean_precision:>8.3f}")
print(f"{'Recall':<20} {run_before.mean_recall:>8.3f} {run_after.mean_recall:>8.3f}")
print(f"{'F1':<20} {run_before.mean_f1:>8.3f} {run_after.mean_f1:>8.3f}")
print(f"{'Omissions':<20} {run_before.total_omissions:>8} {run_after.total_omissions:>8}")
print(f"{'Hallucinations':<20} {run_before.total_hallucinations:>8} {run_after.total_hallucinations:>8}")

Schema validation passed.
Gold validation passed.

Metric                 Before    After
------------------------------------
Precision               0.400    1.000
Recall                  0.400    1.000
F1                      0.400    1.000
Omissions                   0        0
Hallucinations              0        0
